In [66]:
from natasha import Segmenter, MorphVocab, NewsEmbedding, NewsNERTagger
from yargy import rule, or_, Parser
from yargy.pipelines import morph_pipeline
from yargy.predicates import is_capitalized, eq
from yargy.interpretation import fact
import pandas as pd
import re
from pymorphy2 import MorphAnalyzer


In [67]:
file_content = """с. Терновка Ставропольской губ. (ныне с. Урожайное Левокумского р-на Ставропольского края , N44°47'30,80" E44°55'11,08")
с. Большие Вершинята (ныне северная часть с. Вершинята ( Кировская обл.  Уржумский р-н )), 57,263431° 49,893042°
слоб. Песчанка Балашовского уезда Саратовской губ. (ныне Самойловский р-н Саратовской обл. ), 51,308879° 43,676467°
с. Темрянь Белевского уезда Тульской губ. (ныне Белевского р-на Тульской обл. ), 53,765979° 36,214453°
с. Палищи Гусь-Хрустального р-на Владимирской обл.  , 55,427727° 40,322602°
с. Терсалгай (ныне Томская обл. , Кожевниковский р-н ), 56,382599° 83,438522°
г. Холмогоры (ныне село Архангельской обл. , районный центр, 64,224880° 41,651953° )
о. Талабск (Залит), Псковская обл. , Псковский р-н , 58,012314° 28,070124°
станица Ярославская (ныне Краснодарский край , Мостовский р-н ), 44,611150° 40,458970°
пос.Полевский (ныне Свердловская обл. , г. Полевской , N56,435999° E60,187225° )
починок Лощилко Нолинского уезда Вятской губ. , ныне Советского р-на Кировской обл.  , утрачен, N57,702565° E49,207077° (приблизительно)
с.Мормужино Пошехонского уезда Ярославской губ. (ныне Пошехонского р-на Ярославской обл., затоплено Рыбинским водохранилищем), N58,455219° E38,806114° (приблизительно)
д. Полюжье (ныне Угличского р-на Ярославской обл. , в 2 км на З от дер. Прямиково , утрачена, N57,424336° E38,090286°)
с. Яновищи Псковской губ. (ныне Тверская обл. , Андреапольский р-н , N56,854490° E31,506042°)
погост Якотский Дмитровского уезда (ныне с. Якоть Дмитровского р-на Московской обл. , N56,400224° E37,718468°)"""


In [68]:
def dms_to_dd(degrees, minutes, seconds, direction):
    dd = float(degrees) + float(minutes)/60 + float(seconds.replace(',', '.'))/3600
    if direction in ['S', 'W']:
        dd *= -1
    return dd

def dm_to_dd(degrees, minutes, direction):
    dd = float(degrees) + float(minutes.replace(',', '.'))/60
    if direction in ['S', 'W']:
        dd *= -1
    return dd

patterns = [
    r'N(?P<latitude>[-+]?\d{1,2}[.,]\d+°)\s+E(?P<longitude>[-+]?\d{1,3}[.,]\d+°)',
    r'lat=(?P<latitude>[-+]?\d+\.\d+)&long=(?P<longitude>[-+]?\d+\.\d+)',
    r'lat_(?P<latitude>[-+]?\d+\.\d+)&long_(?P<longitude>[-+]?\d+\.\d+)',
    r'Latitude(?P<latitude>[-+]?\d+\.\d+)°Longitude(?P<longitude>[-+]?\d+\.\d+)',
    r'(?P<latitude>[-+]?\d+[.,]\d+)°\s+(?P<longitude>[-+]?\d+[.,]\d+)°'
]

dms_pattern = re.compile(r'(\d{1,3})°\s?(\d{1,2})\'\s?(\d{1,2}(?:[.,]\d+)?)\"\s?([NS]),?\s?(\d{1,3})°\s?(\d{1,2})\'\s?(\d{1,2}(?:[.,]\d+)?)\"\s?([EW])')
dm_pattern = re.compile(r'(\d{1,3})°\s?(\d{1,2}(?:[.,]\d+)?)\'\s?([NS]),?\s?(\d{1,3})°\s?(\d{1,2}(?:[.,]\d+)?)\'\s?([EW])')
dd_pattern = re.compile(r'(-?\d{1,3}[.,]\d+°?[NS]?),?\s?(-?\d{1,3}[.,]\d+°?[EW]?)')
nmea_pattern = re.compile(r'([NS])(\d{1,3})°\s?(\d{1,2})\'\s?(\d{1,2}(?:[.,]\d+)?)\"\s+([EW])(\d{1,3})°\s?(\d{1,2})\'\s?(\d{1,2}(?:[.,]\d+)?)\"')


def extract_coordinates(text):
    coordinates = []

    matches = nmea_pattern.finditer(text)
    for match in matches:
        lat_dir = match.group(1)
        lat_deg = match.group(2)
        lat_min = match.group(3)
        lat_sec = match.group(4)
        lon_dir = match.group(5)
        lon_deg = match.group(6)
        lon_min = match.group(7)
        lon_sec = match.group(8)
        
        lat = dms_to_dd(lat_deg, lat_min, lat_sec, lat_dir)
        lon = dms_to_dd(lon_deg, lon_min, lon_sec, lon_dir)
        
        coordinates.append((lat, lon))
        text = text.replace(match.group(), ' ')

    def replace_with_space(match):
        coordinates.append(match.group())
        return ' '

    text = dms_pattern.sub(replace_with_space, text)
    text = dm_pattern.sub(replace_with_space, text)
    text = dd_pattern.sub(replace_with_space, text)

    for pattern in patterns:
        regex = re.compile(pattern)
        matches = regex.finditer(text)
        for match in matches:
            coordinates.append(match.group())
            text = text.replace(match.group(), ' ')
    
    def convert_to_dd(coord):
        if isinstance(coord, tuple):
            return coord
            
        for pattern in patterns:
            regex = re.compile(pattern)
            match = regex.match(coord)
            if match:
                if all(k in match.groupdict() for k in ('latitude', 'latitude_min', 'latitude_sec', 'longitude', 'longitude_min', 'longitude_sec')):
                    lat = dms_to_dd(match.group('latitude').replace('°', ''), match.group('latitude_min'), match.group('latitude_sec'), 'N')
                    lon = dms_to_dd(match.group('longitude').replace('°', ''), match.group('longitude_min'), match.group('longitude_sec'), 'E')
                    return (lat, lon)
                elif 'latitude' in match.groupdict() and 'longitude' in match.groupdict():
                    lat = match.group('latitude').replace(',', '.').replace('°', '')
                    lon = match.group('longitude').replace(',', '.').replace('°', '')
                    return (float(lat), float(lon))
        
        if 'N' in coord or 'S' in coord or 'E' in coord or 'W' in coord:
            dms_match = dms_pattern.match(coord)
            dm_match = dm_pattern.match(coord)
            dd_match = dd_pattern.match(coord)
            if dms_match:
                lat = dms_to_dd(dms_match.group(1), dms_match.group(2), dms_match.group(3), dms_match.group(4))
                lon = dms_to_dd(dms_match.group(5), dms_match.group(6), dms_match.group(7), dms_match.group(8))
                return (lat, lon)
            elif dm_match:
                lat = dm_to_dd(dm_match.group(1), dm_match.group(2), dm_match.group(3))
                lon = dm_to_dd(dm_match.group(4), dm_match.group(5), dm_match.group(6))
                return (lat, lon)
            elif dd_match:
                lat = float(dd_match.group(1).replace('°N', '').replace('°S', '').replace('N', '').replace('S', ''))
                lon = float(dd_match.group(2).replace('°E', '').replace('°W', '').replace('E', '').replace('W', ''))
                if 'S' in dd_match.group(1):
                    lat = -lat
                if 'W' in dd_match.group(2):
                    lon = -lon
                return (lat, lon)
        return None

    converted_coords = []
    for coord in coordinates:
        converted = convert_to_dd(coord)
        if converted is not None:
            converted_coords.append(converted)
            
    text = text.replace('°', '')
    return text, converted_coords


In [69]:
rows = []
for line in file_content.split('\n'):
    if line.strip():
        cleaned_text, coordinates = extract_coordinates(line)
        
        coord_str = "; ".join([f"{lat}, {lon}" for lat, lon in coordinates]) if coordinates else "Не найдено"
        
        rows.append({
            'Очищенный текст': cleaned_text,
            'Координаты': coord_str
        })

df = pd.DataFrame(rows)

df.head()

,Очищенный текст,Координаты
0,с. Терновка Ставропольской губ. (ныне с. Урожа...,"44.791888888888884, 44.91974444444444"
1,с. Большие Вершинята (ныне северная часть с. В...,"57.263431, 49.893042"
2,слоб. Песчанка Балашовского уезда Саратовской ...,"51.308879, 43.676467"
3,с. Темрянь Белевского уезда Тульской губ. (нын...,"53.765979, 36.214453"
4,с. Палищи Гусь-Хрустального р-на Владимирской ...,"55.427727, 40.322602"


In [70]:
df_missing = df[df['Координаты'] == "Не найдено"]

df_missing

,Очищенный текст,Координаты


In [71]:
abbreviations = {
    'ж.-д. ст.': 'железнодорожная станция',
    'агрогородок': 'агрогородок',
    'ж.-д.': 'железнодорожная',
    'респ.': 'республика',
    'край': 'край',
    'обл.': 'область',

    'а.обл.': 'автономная область',
    'а.окр.': 'автономный округ',
    'м.р-н': 'муниципальный район',
    'г.о.': 'городской округ',
    'г.п.': 'городское поселение',
    'с.п.': 'сельское поселение',
    'пос.': 'посёлок',
    'р-н': 'район',
    'с/с': 'сельсовет',
    'г.': 'город',
    'пгт.': 'поселок городского типа',
    'пос.': 'поселок',
    'аал': 'аал',
    'арбан': 'арбан',
    'аул': 'аул',
    'повят': 'повят',


    "государство": None,
    "автономная территория": None,
    "субъект федерации": None,
    "область": "обл.",
    "край": "кр.",
    "губерния": "губ.",
    "уезд": None,
    "район": "р-н",
    "округ": "окр.",
    "департамент": "деп.",
    "графство": None,
    "жудец": None,
    "комитат": None,
    "воеводство": None,
    "повят": None,
    "город": "г.",
    "муниципалитет": "мун.",
    "гмина": None,
    "коммуна": None,
    "волость": None,
    "автономный округ": "авт. окр.",

    'киш.': 'кишлак',
    'ж/д бл-ст': 'железнодорожный блокпост',
    'ж/д б-ка': 'железнодорожная будка',
    'ж/д в-ка': 'железнодорожная ветка',
    'ж/д к-ма': 'железнодорожная казарма',
    'ж/д к-т': 'железнодорожный комбинат',
    'ж/д пл-ма': 'железнодорожная платформа',
    'ж/д пл-ка': 'железнодорожная площадка',
    'ж/д п.п.': 'железнодорожный путевой пост',
    'ж/д о.п.': 'железнодорожный остановочный пункт',
    'ж/д рзд.': 'железнодорожный разъезд',
    'ж/д ст.': 'железнодорожная станция',
    'м-ко': 'местечко',
    'мест.': 'местечко',

    'с.': 'село',
    'слоб.': 'слобода',
    'ст.': 'станция',
    'ст-ца': 'станица',
    'у.': 'улус',
    'хут.': 'хутор',
    'зим.': 'зимовье',
    'мкр.': 'микрорайон',
    'ост-в': 'остров',
    'о.': 'остров',

    'платф.': 'платформа',

    'ус.': 'усадьба',

    'ю.': 'юрты',
    'ал.': 'аллея',
    'б-р': 'бульвар',
    'взв.': 'взвоз',
    'взд.': 'въезд',
    'дор.': 'дорога',
    'ззд.': 'заезд',
    'км': 'километр',
    'к-цо': 'кольцо',
    'коса': 'коса',
    'лн.': 'линия',
    'мгстр.': 'магистраль',
    'наб.': 'набережная',
    'пер-д': 'переезд',
    'пер.': 'переулок',

    'пл.': 'площадь',
    'пр-д': 'проезд',
    'пр-кт': 'проспект',
    'проул.': 'проулок',
    'туп.': 'тупик',
    'ул.': 'улица',
    'ш.': 'шоссе',
    'влд.': 'владение',
    'зд.': 'здание',
    'з/у': 'земельный участок',
    'кв.': 'квартира',
    'ком.': 'комната',
    'подв.': 'подвал',
    'к.': 'корпус',
    'стр.': 'строение',
    'торг.зал': 'торговый зал',
    'цех': 'цех',
    'губ.': 'губерния',
    'вол.': 'волость',
    'дер.': 'деревня',
    'оз.': 'озеро',
    'хут.': 'хутор',
    'слоб.': 'слобода',
    'вслоб.': 'в слободе',
    'вг.': 'в городе',
    'окр.': 'округ',
    'р-н': 'района',
    'р-на': 'района',
    'р-не': 'района',
    'с.': 'село',
    'обл.': 'область',
    'г.': 'город',
    'обл.': 'область',
    'д.': 'деревня',
    'мкр': 'микрорайон'
}

admin_units = {
    "уезд": "район",
    "губерния": "область",
    "волость": "поселение",
    "р-н": "уезд",
    "обл.": "губ."
}

linear_list = ['починок', 'погост', 'ж.д. ст.']
linear_list.extend([item for pair in abbreviations.items() for item in pair])

In [72]:
segmenter = Segmenter()
emb = NewsEmbedding()
ner_tagger = NewsNERTagger(emb)
morph = MorphAnalyzer()

GeoEntity = fact('GeoEntity', ['type', 'name'])

types_list1 = [
    'слобода', 'района', 'район', 'погоста','область', 'край', 'края', 'губерния', 'уезд', 'волость', 'округ', 'сельсовет', 'АССР'
]
types_list = ['станица', 'город', 'погост', 'погоста', 'деревня', 'хутор', 'станция', 'поселок', 'село', 'республика', 'урочище']

for abbreviation, full_name in abbreviations.items():
    if full_name in types_list1 and abbreviation not in types_list1:
        types_list1.append(abbreviation)
        types_list1.append(abbreviation.lower())
        
    if full_name in types_list and abbreviation not in types_list:
        types_list.append(abbreviation)
        types_list.append(abbreviation.lower())

# Список сокращенных форм
SHORT_FORMS = morph_pipeline(["М.", "Б.", "В.", "Н.", "К.", "С.", "Бол.", "Мал."])

AFTER_SHORT_FORM = rule(SHORT_FORMS, is_capitalized()).interpretation(GeoEntity.name)

AFTER_TYPES = morph_pipeline(types_list).interpretation(GeoEntity.type.normalized())
BEFORE_TYPES = morph_pipeline(types_list1).interpretation(GeoEntity.type.normalized())

GEO_NAME_WITH_HYPHEN = rule(is_capitalized(), eq('-'), is_capitalized()).interpretation(GeoEntity.name)
GEO_NAME_MULTIPLE = rule(is_capitalized().repeatable()).interpretation(GeoEntity.name)
GEO_NAME = or_(GEO_NAME_WITH_HYPHEN, GEO_NAME_MULTIPLE).interpretation(GeoEntity.name)

NAME_BEFORE_TYPE = rule(GEO_NAME, BEFORE_TYPES).interpretation(GeoEntity)
TYPE_BEFORE_NAME = rule(AFTER_TYPES, GEO_NAME).interpretation(GeoEntity)

COMPLEX_NAME = rule(GEO_NAME, eq('-на-'), GEO_NAME).interpretation(GeoEntity.name)
TYPE_BEFORE_SHORT_NAME = rule(morph_pipeline(types_list), AFTER_SHORT_FORM).interpretation(GeoEntity)

SHORT_NAME_ENTITY = AFTER_SHORT_FORM.interpretation(GeoEntity)


GEO_ENTITY = or_(NAME_BEFORE_TYPE, TYPE_BEFORE_NAME, COMPLEX_NAME, TYPE_BEFORE_SHORT_NAME, SHORT_NAME_ENTITY).interpretation(GeoEntity)


In [73]:
parser = Parser(GEO_ENTITY)

extracted_entities = []

for index, row in df.iterrows():
    text = row['Очищенный текст']
    entities = []
    
    for match in parser.findall(text):
        entities.append(str(match.fact))
    
    # Добавляем результат в список
    extracted_entities.append({
        'index': index,
        'text': text,
        'entities': entities,
        'entities_count': len(entities)
    })

entities_df = pd.DataFrame(extracted_entities)

df = df.merge(entities_df[['index', 'entities', 'entities_count']], 
              left_index=True, right_on='index', how='left').drop('index', axis=1)

df.iloc[0]

Очищенный текст    с. Терновка Ставропольской губ. (ныне с. Урожа...
Координаты                     44.791888888888884, 44.91974444444444
entities           [GeoEntity(type='с.', name='Терновка'), GeoEnt...
entities_count                                                     5
Name: 0, dtype: object

In [74]:
from yargy import Parser
import pandas as pd


parser = Parser(GEO_ENTITY)

results = []
for text in df['Очищенный текст']:
    # Удаление части в скобках и всего после
    clean_text = text.split('(')[0].strip()
    matches = list(parser.findall(clean_text))
    pairs = []
    for match in matches:
        entity = match.fact
        pairs.append((entity.type, entity.name))
    results.append(pairs)

df['Извлеченные сущности'] = results
df.head()

,Очищенный текст,Координаты,entities,entities_count,Извлеченные сущности
0,с. Терновка Ставропольской губ. (ныне с. Урожа...,"44.791888888888884, 44.91974444444444","[GeoEntity(type='с.', name='Терновка'), GeoEnt...",5,"[(с., Терновка), (губ., Ставропольской)]"
1,с. Большие Вершинята (ныне северная часть с. В...,"57.263431, 49.893042","[GeoEntity(type='с.', name='Большие Вершинята'...",4,"[(с., Большие Вершинята)]"
2,слоб. Песчанка Балашовского уезда Саратовской ...,"51.308879, 43.676467","[GeoEntity(type='уезд', name='Песчанка Балашов...",4,"[(уезд, Песчанка Балашовского), (губ., Саратов..."
3,с. Темрянь Белевского уезда Тульской губ. (нын...,"53.765979, 36.214453","[GeoEntity(type='с.', name='Темрянь'), GeoEnti...",5,"[(с., Темрянь), (уезд, Белевского), (губ., Тул..."
4,с. Палищи Гусь-Хрустального р-на Владимирской ...,"55.427727, 40.322602","[GeoEntity(type='с.', name='Палищи'), GeoEntit...",3,"[(с., Палищи), (р-на, Гусь-Хрустального), (обл..."


In [75]:
from yargy import Parser
import pandas as pd
import csv
parser = Parser(GEO_ENTITY)

results = []
for index, row in df.iterrows():
    text = row['Очищенный текст']
    clean_text = text.split('(')[0].strip()
    matches = list(parser.findall(clean_text))
    
    entities = []
    for match in matches:
        entity = match.fact
        entities.append(f"{entity.type}:{entity.name}")
    
    results.append({
        'Оригинальный текст': text,
        'Координаты': row['Координаты'],
        'Извлеченные сущности': '; '.join(entities) if entities else 'Не найдено'
    })

result_df = pd.DataFrame(results)

result_df.to_csv('извлеченные_гео_сущности.csv', index=False, encoding='utf-8-sig')


In [76]:
result_df.head()

,Оригинальный текст,Координаты,Извлеченные сущности
0,с. Терновка Ставропольской губ. (ныне с. Урожа...,"44.791888888888884, 44.91974444444444",с.:Терновка; губ.:Ставропольской
1,с. Большие Вершинята (ныне северная часть с. В...,"57.263431, 49.893042",с.:Большие Вершинята
2,слоб. Песчанка Балашовского уезда Саратовской ...,"51.308879, 43.676467",уезд:Песчанка Балашовского; губ.:Саратовской
3,с. Темрянь Белевского уезда Тульской губ. (нын...,"53.765979, 36.214453",с.:Темрянь; уезд:Белевского; губ.:Тульской
4,с. Палищи Гусь-Хрустального р-на Владимирской ...,"55.427727, 40.322602",с.:Палищи; р-на:Гусь-Хрустального; обл.:Владим...


In [77]:
# еще не все правила прописаны

In [78]:
#теперь тоже извлечение ner библиотечно

In [83]:
from natasha import (
    Segmenter,
    MorphVocab,
    NewsEmbedding,
    NewsMorphTagger,
    NewsNERTagger,
    Doc
)
import pandas as pd
import re

segmenter = Segmenter()
morph_vocab = MorphVocab()
emb = NewsEmbedding()
morph_tagger = NewsMorphTagger(emb)
ner_tagger = NewsNERTagger(emb)

def extract_locations(text):
    clean_text = re.split(r'\(|\)', text)[0].strip()
    
    doc = Doc(clean_text)
    doc.segment(segmenter)
    doc.tag_morph(morph_tagger)
    doc.tag_ner(ner_tagger)
    
    locations = []
    for span in doc.spans:
        if span.type == 'LOC':
            span.normalize(morph_vocab)
            locations.append(span.normal)
    
    return locations

df['Географические_названия'] = df['Очищенный текст'].apply(extract_locations)

print("Результаты обработки:")
df.to_csv('результаты_обработки_natasha.csv', index=False, encoding='utf-8-sig')
df.head()

Результаты обработки:


,Очищенный текст,Координаты,entities,entities_count,Извлеченные сущности,Географические_названия
0,с. Терновка Ставропольской губ. (ныне с. Урожа...,"44.791888888888884, 44.91974444444444","[GeoEntity(type='с.', name='Терновка'), GeoEnt...",5,"[(с., Терновка), (губ., Ставропольской)]",[Терновка Ставропольская губы]
1,с. Большие Вершинята (ныне северная часть с. В...,"57.263431, 49.893042","[GeoEntity(type='с.', name='Большие Вершинята'...",4,"[(с., Большие Вершинята)]",[]
2,слоб. Песчанка Балашовского уезда Саратовской ...,"51.308879, 43.676467","[GeoEntity(type='уезд', name='Песчанка Балашов...",4,"[(уезд, Песчанка Балашовского), (губ., Саратов...","[Песчанка, Балашовский уезд, Саратовская]"
3,с. Темрянь Белевского уезда Тульской губ. (нын...,"53.765979, 36.214453","[GeoEntity(type='с.', name='Темрянь'), GeoEnti...",5,"[(с., Темрянь), (уезд, Белевского), (губ., Тул...","[Темрянь Белевский уезд, Тульская]"
4,с. Палищи Гусь-Хрустального р-на Владимирской ...,"55.427727, 40.322602","[GeoEntity(type='с.', name='Палищи'), GeoEntit...",3,"[(с., Палищи), (р-на, Гусь-Хрустального), (обл...","[Гусь-Хрустального р-на, Владимирская обл]"


In [65]:
#сравнение результатов

| Извлечённые сущности(библиотека) | Извлечённые сущности(написанными правилами) |
|------------------------|----------------------|
| Терновка Ставропольская губы | с.:Терновка губ.:Ставропольской |
| Песчанка, Балашовский уезд, Саратовская | уезд:Песчанка Балашовского губ.:Саратовской |
| Темрянь Белевский уезд, Тульская | с.:Темрянь уезд:Белевского губ.:Тульской |
| Гусь-Хрустального р-на, Владимирская обл | с.:Палищи р-на:Гусь-Хрустального обл.:Владимирской |
| Холмогоры | г.:Холмогоры |
| Ярославская | станица:Ярославская |
| Лощилко Нолинский уезд, Вятская губ, Советский р-на, Кировская обл | уезд:Лощилко Нолинского губ.:Вятской р-на:Советского обл.:Кировской |
| Мормужино Пошехонский уезд, Ярославская | с.:Мормужино уезд:Пошехонского губ.:Ярославской |
| Якотский, Дмитровский уезд | погост:Якотский уезд:Дмитровского |
| — | с.:Терсалгай |
| — | пос.:Полевский |
| — | д.:Полюжье |
| — | с.:Яновищи губ.:Псковской |

In [4]:
# **Функция обработки текста**
def process_text(text):
    geo_parser = Parser(GEO_ENTITY)
    historical_parser = Parser(HISTORICAL_CHANGE)

    geo_matches = [match.fact for match in geo_parser.findall(text) if match.fact]
    historical_matches = [match.fact for match in historical_parser.findall(text) if match.fact]

    return geo_matches, historical_matches



In [6]:
for line in file_content.split('\n'):
    line = line.strip()
    if line:
        geo_result, historical_result = process_text(line)

        print(f"Исходный текст: {line}")

        print("\n📍 Географические объекты:")
        for entity in geo_result:
            print(f"Тип: {getattr(entity, 'type', None)}, Название: {getattr(entity, 'name', None)}, Альтернативное название: {getattr(entity, 'alt_name', None)}")

        print("\n📜 Исторические изменения:")
        for entity in historical_result:
            print(f"Тип: {entity.admin_type}, Текущий регион: {entity.current}, Историческое название: {entity.historical}")
        print()

Исходный текст: с. Терновка Ставропольской губ. (ныне с. Урожайное Левокумского р-на Ставропольского края, N44°47'30,80" E44°55'11,08")

📍 Географические объекты:

📜 Исторические изменения:

Исходный текст: с. Большие Вершинята (ныне северная часть с. Вершинята (Кировская обл., Уржумский р-н)), 57,263431° 49,893042°

📍 Географические объекты:
Тип: р-н, Название: Уржумский, Альтернативное название: None

📜 Исторические изменения:

Исходный текст: слоб. Песчанка Балашовского уезда Саратовской губ. (ныне Самойловский р-н Саратовской обл.), 51,308879° 43,676467°

📍 Географические объекты:
Тип: уезд, Название: Песчанка Балашовского, Альтернативное название: None
Тип: р-н, Название: Самойловский, Альтернативное название: None

📜 Исторические изменения:

Исходный текст: с. Темрянь Белевского уезда Тульской губ. (ныне Белевского р-на Тульской обл.), 53,765979° 36,214453°

📍 Географические объекты:
Тип: уезд, Название: Темрянь Белевского, Альтернативное название: None

📜 Исторические изменения: